In [22]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# ==========================================
# 1. Load the Original Data
# ==========================================
# Required Variables
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Drop both Revenue AND Session_ID from the training features
X_train = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
y_train = original_df['Revenue']

# ==========================================
# 2. Impute Missing Values Safely with Pandas
# ==========================================
# Automatically identify columns (Updated to handle Pandas 3.0 string types)
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# LEARN the medians and modes exclusively from the training set
train_medians = X_train[numeric_features].median()
train_modes = X_train[categorical_features].mode().iloc[0]

# APPLY the imputations to X_train
X_train_imputed = X_train.copy()
X_train_imputed[numeric_features] = X_train_imputed[numeric_features].fillna(train_medians)
X_train_imputed[categorical_features] = X_train_imputed[categorical_features].fillna(train_modes)

# ==========================================
# 3. Scale and Encode 
# ==========================================
# Create a ColumnTransformer WITHOUT SimpleImputer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# Apply transformations
X_train_processed_array = preprocessor.fit_transform(X_train_imputed)
feature_names = preprocessor.get_feature_names_out()

# Required Variable: Reconstruct the processed dataframe
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

# ==========================================
# 4. Train the Model
# ==========================================
# Required Variable
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']
model.fit(X_processed, y_processed)

# ==========================================
# 5. Prepare Test Data and Generate Predictions
# ==========================================
# Drop Session_ID from test set features
X_test = test_df.drop(columns=['Session_ID'])

# Apply the exact same imputations learned from X_train to X_test
X_test_imputed = X_test.copy()
X_test_imputed[numeric_features] = X_test_imputed[numeric_features].fillna(train_medians)
X_test_imputed[categorical_features] = X_test_imputed[categorical_features].fillna(train_modes)

# Transform the test data
X_test_processed_array = preprocessor.transform(X_test_imputed)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)

# Required Variable
predictions = model.predict(X_test_processed)

In [23]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Checkpoint information
# ---------------------------------------------------

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100


# ---------------------------------------------------
# Detect model used
# ---------------------------------------------------

final_model = model

# Handle GridSearchCV / RandomizedSearchCV
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

# Handle sklearn Pipeline
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__


# ---------------------------------------------------
# Create checkpoint section
# ---------------------------------------------------

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})


# ---------------------------------------------------
# Create prediction section
# ---------------------------------------------------

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})


# ---------------------------------------------------
# Combine and save
# ---------------------------------------------------

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)


submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  30
6  row_retained_percent               100.0
7            model_name  LogisticRegression
